# Final Performance Benchmark — All 3 Tiers Combined

**Purpose:** Comprehensive E2E performance test after ALL optimization tiers.

## Optimization Summary

### Tier 1 — Quick Wins (commit `4c1464b`)
- **T-OPT-1.1:** Parallel parse batches (`asyncio.gather` + `Semaphore(3)`)
- **T-OPT-1.2:** Cache Vertex AI Search context on retry (skip `retrieve_context()`)
- **T-OPT-1.3:** Skip supervisor on retry (`fail → math_agent`)

### Tier 2 — Medium Optimizations
- **T-OPT-2.1:** Parse-only retry (cache raw text, retry parsing before re-generating)
- **T-OPT-2.3:** Parallel micro-batch generation (split large batches into 2 concurrent calls)
- **T-OPT-2.2:** SKIPPED — Gemini 3 Flash infinite loop with strict JSON schemas

### Tier 3 — Architectural
- **T-OPT-3.2:** SDK migration to `langchain-google-genai` (already done 2026-03-27)
- **T-OPT-3.3:** Deterministic supervisor (`USE_LLM_CLASSIFICATION=False`, skip LLM call)
- **T-OPT-3.4:** Parallel formatter batches (`asyncio.gather` for quiz/flashcard/fill_blank)
- **T-OPT-3.1:** Pre-generate pool — BLOCKED (requires Firestore + Cloud Tasks infrastructure)

## Baseline (pre-optimization, 2026-04-09)

| Difficulty | Time | Iterations |
|---|---|---|
| recall | 125.1s | 1 |
| comprehension | 122.7s | 1 |
| application | 539.9s | 2 |
| high_application | 1290.8s | 4 |
| **TOTAL** | **2078.5s** | - |

## Targets

| Tier | Target |
|---|---|
| Tier 1 | high_application ≤ 800s, total ≤ 1400s |
| Tier 2 | high_application ≤ 400s, total ≤ 900s |
| Combined | 100% delivery (40/40 items) |

## Section 1: Setup & Configuration Verification

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
os.environ.setdefault('ENV', 'develop')

from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope, GameContentResponse
)
from src.graph.builder import compile_graph
from src.graph.nodes.math_agent import (
    MICRO_BATCH_THRESHOLD, MAX_PARSE_ONLY_RETRIES,
    MAX_GENERATION_RETRIES, MAX_PARSE_RETRIES, PARSE_BATCH_SIZE,
)
from src.graph.nodes.supervisor import USE_LLM_CLASSIFICATION
from src.config.constants import FORMATTER_BATCH_SIZE

print('=' * 60)
print('CONFIGURATION VERIFICATION')
print('=' * 60)
print(f'Tier 1:')
print(f'  PARSE_BATCH_SIZE = {PARSE_BATCH_SIZE}')
print(f'  MAX_PARSE_RETRIES = {MAX_PARSE_RETRIES}')
print(f'Tier 2:')
print(f'  MAX_PARSE_ONLY_RETRIES = {MAX_PARSE_ONLY_RETRIES}')
print(f'  MAX_GENERATION_RETRIES = {MAX_GENERATION_RETRIES}')
print(f'  MICRO_BATCH_THRESHOLD = {MICRO_BATCH_THRESHOLD}')
print(f'Tier 3:')
print(f'  USE_LLM_CLASSIFICATION = {USE_LLM_CLASSIFICATION}')
print(f'  FORMATTER_BATCH_SIZE = {FORMATTER_BATCH_SIZE}')
print(f'\nAll imports OK ✅')

## Section 2: SDK & Architecture Verification

In [ ]:
import inspect
import src.services.llm as llm_mod
import src.graph.nodes.supervisor as sup_mod
import src.graph.nodes.formatter as fmt_mod

# T-OPT-3.2: Verify SDK migration
llm_src = inspect.getsource(llm_mod)
assert 'ChatGoogleGenerativeAI' in llm_src, 'Expected ChatGoogleGenerativeAI in llm.py'
assert 'ChatVertexAI' not in llm_src, 'ChatVertexAI should not be in llm.py'
print('✅ T-OPT-3.2: ChatGoogleGenerativeAI in use, no ChatVertexAI')

# T-OPT-3.3: Verify deterministic supervisor
assert not USE_LLM_CLASSIFICATION, 'Expected USE_LLM_CLASSIFICATION=False'
print(f'✅ T-OPT-3.3: USE_LLM_CLASSIFICATION = {USE_LLM_CLASSIFICATION}')

# T-OPT-3.4: Verify parallel formatter (asyncio.gather in format functions)
fmt_src = inspect.getsource(fmt_mod)
gather_count = fmt_src.count('asyncio.gather')
assert gather_count >= 3, f'Expected ≥3 asyncio.gather calls in formatter, got {gather_count}'
print(f'✅ T-OPT-3.4: {gather_count} asyncio.gather calls in formatter (quiz/flashcard/fill_blank)')

# T-OPT-1.3: Verify graph topology
from src.graph.builder import compile_graph
graph = compile_graph()
print(f'✅ Graph compiled successfully')

print('\n🔬 All architecture checks passed')

## Section 3: E2E Runner

In [ ]:
NUM_QUESTIONS = 10
TOPIC = "Đạo hàm và ứng dụng"

# Baseline numbers (pre-optimization, E2E 2026-04-09)
BASELINE = {
    "recall":           {"time": 125.1, "iterations": 1},
    "comprehension":    {"time": 122.7, "iterations": 1},
    "application":      {"time": 539.9, "iterations": 2},
    "high_application": {"time": 1290.8, "iterations": 4},
}
BASELINE_TOTAL = sum(b["time"] for b in BASELINE.values())

# Performance targets
TARGETS = {
    "tier1": {"ha_max": 800, "total_max": 1400},
    "tier2": {"ha_max": 400, "total_max": 900},
}


async def run_e2e(difficulty: str):
    """Run pipeline end-to-end and return (num_delivered, elapsed, iterations, errors)."""
    req = GenerationRequest(
        user_id="perf_benchmark_final",
        topic=TOPIC,
        game_types=[GameType.QUIZ],
        num_questions=NUM_QUESTIONS,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope.SYSTEM,
    )
    app = compile_graph()
    initial_state = {
        "request": req,
        "doc_scope": req.doc_scope.value,
        "iteration_count": 0,
        "rejected_items": [],
        "reviewed_items": [],
        "errors": [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0

    output: GameContentResponse | None = result.get("final_output")
    delivered = len(output.content.quiz) if output else 0
    iterations = result.get("iteration_count", 0)
    errors = result.get("errors", [])
    return delivered, elapsed, iterations, errors


print(f"Runner ready — Topic: {TOPIC} | {NUM_QUESTIONS} questions per level")
print(f"Baseline total: {BASELINE_TOTAL:.1f}s")

## Section 4: Run Full Benchmark (All 4 Difficulty Levels)

In [ ]:
print("=" * 80)
print("FINAL PERFORMANCE BENCHMARK — ALL TIERS COMBINED")
print(f"Topic: {TOPIC} | Requested: {NUM_QUESTIONS} quiz per level")
print("=" * 80)

e2e_results = {}
total_start = time.time()

for diff in ["recall", "comprehension", "application", "high_application"]:
    print(f"\n--- Running {diff} ---")
    delivered, elapsed, iters, errors = await run_e2e(diff)
    pct = delivered / NUM_QUESTIONS * 100
    status = "✅" if delivered >= NUM_QUESTIONS else "❌"
    baseline_t = BASELINE[diff]["time"]
    speedup = (baseline_t - elapsed) / baseline_t * 100
    print(f"  {status} Delivered: {delivered}/{NUM_QUESTIONS} ({pct:.0f}%) in {elapsed:.1f}s ({iters} iter)")
    print(f"  ⏱️  Baseline: {baseline_t:.1f}s → Now: {elapsed:.1f}s ({speedup:+.1f}%)")
    if errors:
        print(f"  ⚠️ Errors: {errors}")
    e2e_results[diff] = {
        "delivered": delivered,
        "requested": NUM_QUESTIONS,
        "pct": pct,
        "elapsed": elapsed,
        "iterations": iters,
        "passed": delivered >= NUM_QUESTIONS,
        "baseline": baseline_t,
        "speedup_pct": speedup,
    }

wall_clock = time.time() - total_start
print(f"\n--- Wall clock: {wall_clock:.1f}s ---")

## Section 5: Comprehensive Verdict

In [ ]:
print("\n" + "=" * 95)
print("FINAL PERFORMANCE BENCHMARK — COMPREHENSIVE RESULTS")
print("=" * 95)
print(f"{'Difficulty':<20} {'Delivered':>9} {'Rate':>7} {'Baseline':>9} {'Now':>8} {'Speedup':>9} {'Iters':>6} {'Status':>8}")
print("-" * 95)

all_passed = True
for diff, r in e2e_results.items():
    status = "✅ PASS" if r["passed"] else "❌ FAIL"
    if not r["passed"]:
        all_passed = False
    print(
        f"{diff:<20} {r['delivered']:>5}/{r['requested']:<3} {r['pct']:>6.0f}% "
        f"{r['baseline']:>8.1f}s {r['elapsed']:>7.1f}s {r['speedup_pct']:>+8.1f}% "
        f"{r['iterations']:>5} {status:>8}"
    )

print("-" * 95)
total_d = sum(r["delivered"] for r in e2e_results.values())
total_r = sum(r["requested"] for r in e2e_results.values())
total_pct = total_d / total_r * 100 if total_r else 0
total_time = sum(r["elapsed"] for r in e2e_results.values())
total_speedup = (BASELINE_TOTAL - total_time) / BASELINE_TOTAL * 100
total_iters = sum(r["iterations"] for r in e2e_results.values())
print(
    f"{'TOTAL':<20} {total_d:>5}/{total_r:<3} {total_pct:>6.0f}% "
    f"{BASELINE_TOTAL:>8.1f}s {total_time:>7.1f}s {total_speedup:>+8.1f}% "
    f"{total_iters:>5}"
)

# ─── Target Verification ────────────────────────────────────────
ha_time = e2e_results["high_application"]["elapsed"]

print("\n" + "=" * 95)
print("PERFORMANCE TARGETS")
print("=" * 95)

# Delivery
delivery_ok = all_passed
print(f"  {'✅' if delivery_ok else '❌'} 100% delivery: {total_d}/{total_r} ({total_pct:.0f}%)")

# Tier 1 targets
t1_ha = TARGETS['tier1']['ha_max']
t1_total = TARGETS['tier1']['total_max']
t1_ha_ok = ha_time <= t1_ha
t1_total_ok = total_time <= t1_total
print(f"  {'✅' if t1_ha_ok else '⚠️'} [Tier 1] high_application ≤ {t1_ha}s: {ha_time:.1f}s")
print(f"  {'✅' if t1_total_ok else '⚠️'} [Tier 1] Total ≤ {t1_total}s: {total_time:.1f}s")

# Tier 2 targets
t2_ha = TARGETS['tier2']['ha_max']
t2_total = TARGETS['tier2']['total_max']
t2_ha_ok = ha_time <= t2_ha
t2_total_ok = total_time <= t2_total
print(f"  {'✅' if t2_ha_ok else '⚠️'} [Tier 2] high_application ≤ {t2_ha}s: {ha_time:.1f}s")
print(f"  {'✅' if t2_total_ok else '⚠️'} [Tier 2] Total ≤ {t2_total}s: {total_time:.1f}s")

# ─── Optimization Breakdown ─────────────────────────────────────
print("\n" + "=" * 95)
print("OPTIMIZATION BREAKDOWN")
print("=" * 95)
print("  Tier 1 (Quick Wins):")
print(f"    T-OPT-1.1: Parallel parse batches (PARSE_BATCH_SIZE={PARSE_BATCH_SIZE})")
print(f"    T-OPT-1.2: Cache Vertex AI Search on retry")
print(f"    T-OPT-1.3: Skip supervisor on retry (fail → math_agent)")
print("  Tier 2 (Medium):")
print(f"    T-OPT-2.1: Parse-only retry (MAX_PARSE_ONLY_RETRIES={MAX_PARSE_ONLY_RETRIES})")
print(f"    T-OPT-2.3: Parallel micro-batch (MICRO_BATCH_THRESHOLD={MICRO_BATCH_THRESHOLD})")
print(f"    T-OPT-2.2: SKIPPED (Gemini 3 Flash JSON mode infinite loop)")
print("  Tier 3 (Architectural):")
print(f"    T-OPT-3.2: SDK migration — ChatGoogleGenerativeAI ✅")
print(f"    T-OPT-3.3: Deterministic supervisor (USE_LLM_CLASSIFICATION={USE_LLM_CLASSIFICATION})")
print(f"    T-OPT-3.4: Parallel formatter batches (FORMATTER_BATCH_SIZE={FORMATTER_BATCH_SIZE})")
print(f"    T-OPT-3.1: Pre-gen pool — BLOCKED (needs infrastructure)")

# ─── Final Verdict ───────────────────────────────────────────────
print("\n" + "=" * 95)
if delivery_ok:
    print(f"🎯 100% delivery maintained | Total speedup: {total_speedup:+.1f}% ({BASELINE_TOTAL:.0f}s → {total_time:.0f}s)")
else:
    failed = [d for d, r in e2e_results.items() if not r["passed"]]
    print(f"❌ DELIVERY REGRESSION — Failed: {', '.join(failed)}")
print("=" * 95)

## Section 6: Export Results (Optional)

In [ ]:
# Save results to JSON for tracking
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "topic": TOPIC,
    "num_questions": NUM_QUESTIONS,
    "tiers": ["tier1", "tier2", "tier3"],
    "baseline_total_s": BASELINE_TOTAL,
    "optimized_total_s": round(total_time, 1),
    "speedup_pct": round(total_speedup, 1),
    "delivery_rate": f"{total_d}/{total_r}",
    "all_passed": all_passed,
    "per_difficulty": {
        diff: {
            "delivered": r["delivered"],
            "elapsed_s": round(r["elapsed"], 1),
            "baseline_s": r["baseline"],
            "speedup_pct": round(r["speedup_pct"], 1),
            "iterations": r["iterations"],
        }
        for diff, r in e2e_results.items()
    },
    "config": {
        "PARSE_BATCH_SIZE": PARSE_BATCH_SIZE,
        "MAX_PARSE_ONLY_RETRIES": MAX_PARSE_ONLY_RETRIES,
        "MAX_GENERATION_RETRIES": MAX_GENERATION_RETRIES,
        "MICRO_BATCH_THRESHOLD": MICRO_BATCH_THRESHOLD,
        "USE_LLM_CLASSIFICATION": USE_LLM_CLASSIFICATION,
        "FORMATTER_BATCH_SIZE": FORMATTER_BATCH_SIZE,
    },
}

print(json.dumps(report, indent=2, ensure_ascii=False))